# Model inference service example

This notebook allows you to:
1. Connect to the R2 training/inference backend.
2. Start, view, and remove inference services.

Uses the `ModelServicesClient` from the main SDK client module (`r2_labs.sdk.client`).

Remember to start the R2 backend first before running this notebook.

In [ ]:
import dotenv

from r2_labs.rpc import client as rpc_client
from r2_labs.sdk import client as sdk_client
from r2_labs.sdk import logging as r2_logging
from r2_labs.sdk import rpc_api
from r2_labs.sdk import sentry

dotenv.load_dotenv()
r2_logging.configure(service="model-services")
sentry.init_sentry(service="model-services")

%load_ext autoreload
%autoreload 2

## 1. Configure Server Connection

Set the hostname/IP of the machine running the R2 backend.

In [ ]:
# Configure the server address
TRAINING_SERVER_HOST = "localhost"  # Change to your server IP/hostname
TRAINING_SERVER_PORT = rpc_api.DEFAULT_MODEL_TRAINER_PORT  # 7534

server_address = f"tcp://{TRAINING_SERVER_HOST}:{TRAINING_SERVER_PORT}"
print(f"Training server: {server_address}")

## 2. Connect to the Server

Create a `ModelServicesClient` using the SDK client module.

In [ ]:
# Create the base RPC client
base_client = rpc_client.BaseClient(
    server_address,
    timeout=30000,  # 30 second timeout
)

# Create the model services client from the SDK
models_client = sdk_client.ModelServicesClient(base_client)

# Test connection by getting status
try:
    status = models_client.get_all()
    print("Connected to Model Services server!")
except Exception as e:
    print(f"Failed to connect: {e}")

## 3. List all currently running model services.

In [ ]:
running_models = models_client.get_all()
print("Running model services:")
for model in running_models:
  print(f"- {model.model_id} (health?: {model.healthy})")

## 4. Start a model service

In [ ]:
MODEL_ID = "rectify_skill_remove_plate#primary-mean-923"
models_client.start(
  model_id=MODEL_ID,
)
models_client.wait_until_ready()  # waits until all models are ready.
address = models_client.get_address(MODEL_ID)
print("Model service is ready!")
print("Running at address:", address)

## 5. Stop the model service

Important to do to free up GPU memory.

In [ ]:
models_client.stop_all()